# Kazakh profanity dataset — сборка из KazSAnDRA

Шаги: загрузка KazSAnDRA → regex-разметка (`kz_regex.py`) → отбор выборки → итоговые метки
(regex + исправления LLM из `llm_corrections.csv`) → удаление личных данных → дубли и почти-дубли →
train/val/test → сохранение.

Рядом с ноутбуком должны лежать **`kz_regex.py`** и **`llm_corrections.csv`**.
В Colab загрузите их в `/content` (значок папки слева → Upload) или подключите Google Drive.

In [1]:
!pip install -q datasketch
!git clone -q --depth 1 https://github.com/IS2AI/KazSAnDRA
!cd KazSAnDRA/dataset && unzip -o -q 06_sc_train_ib.zip && unzip -o -q 09_sc_valid.zip && unzip -o -q 10_sc_test.zip

In [2]:
import re
from pathlib import Path

import pandas as pd
from datasketch import MinHash, MinHashLSH
from sklearn.model_selection import StratifiedGroupKFold

import kz_regex

SEED = 42
KAZSANDRA = Path("KazSAnDRA/dataset")
OUT = Path("data"); OUT.mkdir(exist_ok=True)

## 1. Загрузка KazSAnDRA

In [3]:
files = ["06_sc_train_ib.csv", "09_sc_valid.csv", "10_sc_test.csv"]
df = pd.concat([pd.read_csv(KAZSANDRA / f) for f in files], ignore_index=True)
df = df.rename(columns={"custom_id": "source_id"})
df["rating"] = df["label"] + 1          # в KazSAnDRA оценки 1-5 записаны как 0-4
df["source"] = "kazsandra"
df = df[["source", "source_id", "domain", "rating", "text"]]
print(len(df))
df.domain.value_counts()

175158


domain
appstore     132573
market        29097
mapping        8492
bookstore      4996
Name: count, dtype: int64

## 2. Regex-разметка всех отзывов
`version=1` — словарь, которым отбиралась выборка (для воспроизводимости). Для новых данных используйте `version=2`.

In [4]:
df = df.join(kz_regex.tag_series(df.text, version=1))
print(df.regex_label.value_counts().rename({1: "мат", -1: "спорно", 0: "нет"}))
kz_regex.rule_counts(df.regex_terms).head(20)

regex_label
нет       173376
мат         1670
спорно       112
Name: count, dtype: int64


regex_terms
kk_qotaq     958
ru_blya      196
ru_pizd      152
ru_suka      114
ru_xuj       101
kk_sik        98
ru_eb         65
ru_govno      60
kk_boq        55
ru_euph       51
ru_pidor      48
kk_boqtau     42
kk_kot        42
kk_am         26
ru_lat        21
ru_gandon     18
kk_zhalap     16
m_qotaq        3
m_xuj          3
ru_her         3
Name: count, dtype: int64

In [5]:
# для сравнения: сколько находит улучшенная версия 2
(kz_regex.tag_series(df.text, version=2).regex_label == 1).sum()

np.int64(2579)

## 3. Отбор выборки для разметки
Точные дубли удаляются; берём все срабатывания regex + 1000 случайных «чистых» отзывов с оценкой 1-2 + 1500 случайных «чистых» остальных.

In [6]:
df["norm"] = df.text.astype(str).map(lambda t: re.sub(r"[^\w ]", "", kz_regex.normalize(t)).strip())
print("до удаления точных дублей:", len(df))
df = df.drop_duplicates("norm")
print("после:", len(df))

hits = df[df.regex_label != 0].assign(sampling=lambda x: x.regex_label.map({1: "regex_hit", -1: "regex_weak"}))
neg = df[df.regex_label == 0]
low = neg[neg.rating <= 2].sample(1000, random_state=SEED).assign(sampling="random_low_rating")
rest = neg.drop(low.index).sample(1500, random_state=SEED).assign(sampling="random")

pool = pd.concat([hits, low, rest]).sample(frac=1, random_state=SEED).reset_index(drop=True)
pool["id"] = ["kz" + str(i).zfill(5) for i in range(len(pool))]
pool = pool.drop(columns="norm")
pool.sampling.value_counts()

до удаления точных дублей: 175158
после: 170595


sampling
regex_hit            1618
random               1500
random_low_rating    1000
regex_weak            111
Name: count, dtype: int64

## 4. Итоговые метки
`label = 1`, если regex v1 дал 1, иначе 0. Затем применяются исправления из проверки LLM
(`llm_corrections.csv`: 1 / 0 / EXCLUDE, с причиной).

In [7]:
corr = pd.read_csv("llm_corrections.csv", dtype=str).set_index("source_id")

pool["label"] = (pool.regex_label == 1).astype(int)
pool["label_source"] = pool.regex_label.map({1: "regex_confirmed_by_llm",
                                             -1: "llm_resolved_weak",
                                             0: "regex_negative_llm_spotchecked"})
m = pool.source_id.isin(corr.index)
fl = pool.loc[m, "source_id"].map(corr.final_label)
pool.loc[m, "label_source"] = fl.map(lambda x: "excluded_ambiguous" if x == "EXCLUDE" else "llm_corrected")
pool.loc[m & (fl != "EXCLUDE"), "label"] = fl[fl != "EXCLUDE"].astype(int)
pool = pool[pool.label_source != "excluded_ambiguous"].copy()

print(pool.label.value_counts())
print(pool.label_source.value_counts())

label
0    2564
1    1659
Name: count, dtype: int64
label_source
regex_negative_llm_spotchecked    2469
regex_confirmed_by_llm            1603
llm_resolved_weak                   85
llm_corrected                       66
Name: count, dtype: int64


In [8]:
# негативный отзыв ≠ мат: метка не совпадает с оценкой
pd.crosstab(pool.rating, pool.label, margins=True)

label,0,1,All
rating,,,
1,1059,879,1938
2,209,48,257
3,74,31,105
4,83,39,122
5,1139,662,1801
All,2564,1659,4223


In [9]:
# насколько regex v1 совпадает с итоговыми метками
h = pool.regex_label == 1
print("precision regex v1:", round((h & (pool.label == 1)).sum() / h.sum(), 3))
print("пропущено regex v1:", ((~h) & (pool.label == 1)).sum(), "из", (~h).sum())

precision regex v1: 0.994
пропущено regex v1: 56 из 2610


## 5. Удаление личных данных

In [10]:
UP, LO = "А-ЯЁӘҒҚҢӨҰҮҺІ", "а-яёәғқңөұүһі"
SURNAME = rf"[{UP}][{LO}]+(?:ов|ова|ев|ева|ёв|ёва|ұлы|улы|қызы|кызы|енко)"
MANUAL_NAMES = ["Ернар Тұрсын", "Сымбат Еркін", "Мұрат Әбдлдә", "Мади Бекжігіт", "Медет Бүркітбай",
                "Жания Ақылбек", "Ұлдана Көшкінбаев", "Баян Жусупбекова", "Alan Warmer", "Aizere love"]
KEEP = {"Путин", "Берлин", "Пекин", "Кремлин", "Ленин", "Сталин", "Пушкин", "Абай", "Каспий"}
PII = [
    (re.compile("|".join(re.escape(n) for n in MANUAL_NAMES), re.I), "<NAME>"),
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"), "<EMAIL>"),
    (re.compile(r"https?://\S+|www\.\S+"), "<URL>"),
    (re.compile(r"(?:\+?7|8)[\s\-()]*\d{3}[\s\-()]*\d{3}[\s\-]*\d{2}[\s\-]*\d{2}"), "<PHONE>"),
    (re.compile(r"\b\d{12}\b"), "<ID>"),
    (re.compile(r"\b(?:\d[ -]?){16}\b"), "<CARD>"),
    (re.compile(r"(?<!\w)@[\w.\-]+"), "<USER>"),
    (re.compile(rf"\b[{UP}][{LO}]{{2,}}\s+(?:{SURNAME}|[{UP}][{LO}]+(?:бай|бек|бекова|баев|баева|ев|ов))\b"), "<NAME>"),
    (re.compile(rf"\b[{UP}]{{3,}}\s+[{UP}]{{3,}}(?:ОВА|ОВ|ЕВА|ЕВ|БЕК|БЕКОВА|БАЙ|ҰЛЫ|ҚЫЗЫ)\b"), "<NAME>"),
    (re.compile(rf"\b{SURNAME}\b"), "<NAME>"),
]

def scrub(text):
    text = str(text)
    for rx, tok in PII:
        if tok == "<NAME>":   # не затираем мат, похожий на фамилию
            text = rx.sub(lambda m: m.group(0) if (m.group(0) in KEEP or kz_regex.tag(m.group(0))[0] != 0) else tok, text)
        else:
            text = rx.sub(tok, text)
    return text

new = pool.text.map(scrub)
print("изменено текстов:", (new != pool.text.astype(str)).sum())
pool["text"] = new

изменено текстов: 22


## 6. Дубли и почти-дубли
Почти-дубли объединяются в группы (`group_id`) и всегда попадают в один сплит:
(а) MinHash LSH по символьным 5-граммам + точный Jaccard ≥ 0.8;
(б) одинаковый «грубый ключ» (каз. буквы → рус., без пробелов и знаков, без повторов букв).
Здесь ничего не обучается на данных — это фиксированные правила и хеш-функции.

In [11]:
def norm_key(t):
    t = re.sub(r"[^\w\s]", " ", kz_regex.normalize(t))
    return re.sub(r"\s+", " ", t).strip()

FOLD = str.maketrans("әғқңөұүһіыйёэщц", "агкноуухиииеешс")
def fuzzy_key(t):
    t = re.sub(r"[^a-zа-я]", "", t.translate(FOLD))
    return re.sub(r"(.)\1+", r"\1", t)

def shingles(t, n=5):
    t = f" {t} "
    return {t} if len(t) <= n else {t[i:i + n] for i in range(len(t) - n + 1)}

pool["key"] = pool.text.astype(str).map(norm_key)
pool = pool[pool.key.str.len() > 0]

# точные дубли (если у дублей разные метки — удаляем все)
conflict = pool.groupby("key").label.nunique()
pool = pool[~pool.key.isin(conflict[conflict > 1].index)]
pool = pool.drop_duplicates("key").reset_index(drop=True)

# union-find
parent = list(range(len(pool)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b): parent[find(a)] = find(b)

sh = [shingles(t) for t in pool.key]
lsh, mhs = MinHashLSH(threshold=0.8, num_perm=128), []
for i, s in enumerate(sh):
    mh = MinHash(num_perm=128, seed=SEED)
    for g in s: mh.update(g.encode("utf-8"))
    lsh.insert(str(i), mh); mhs.append(mh)
links = 0
for i, mh in enumerate(mhs):
    for j in map(int, lsh.query(mh)):
        if j > i and len(sh[i] & sh[j]) / len(sh[i] | sh[j]) >= 0.8:
            union(i, j); links += 1
first = {}
for i, fk in enumerate(pool.key.map(fuzzy_key)):
    if fk in first: union(i, first[fk]); links += 1
    else: first[fk] = i

pool["group_id"] = [find(i) for i in range(len(pool))]
sizes = pool.group_id.value_counts()
print("записей:", len(pool), "| связей почти-дублей:", links,
      "| групп >1:", (sizes > 1).sum(), "| в них записей:", sizes[sizes > 1].sum())

записей: 4223 | связей почти-дублей: 65 | групп >1: 43 | в них записей: 108


In [12]:
# пример групп почти-дублей
for gid in sizes[sizes > 1].index[:5]:
    print(pool[pool.group_id == gid].text.str[:70].tolist())

['Қотақ баас ойын', 'Котакбас оиын 🤣🤣🤣🤣🤣', 'Қотақбас ойын', 'Котак бас ойын', 'Қотақ бас ойын 😡🖕🖕😑😑😑', 'Котакбас ойын']
['Қотақбастар', 'Котак бастар', 'Котакбастар', 'Қотақ бастар', 'Қотақ бас тар']
['Пиздес ойын екен', 'Пиздес оиын екен🤣🤣🤣🤣🤣🤣🤣😂😂😂😂😅😅😅😅', 'Пиздес ойн екен', 'Пиздец оиын екен', 'Пиздец ойын екен']
['Қотақ бас', 'Котакбас', 'Қотак бас', 'Қотақбассссс', 'Котак бас']
['Ө-өте керемет', 'Өте керееемеетт', 'Өте керемет ( ˘ ³˘)❤', 'өтекерем ет']


## 7. Train / val / test
`StratifiedGroupKFold(10, shuffle=True, random_state=42)`: фолд 0 → test, фолд 1 → val, остальные → train (≈80/10/10), стратификация по метке, группы не пересекают сплиты.

In [13]:
folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
             .split(pool, pool.label, pool.group_id))
pool["split"] = "train"
pool.loc[folds[0][1], "split"] = "test"
pool.loc[folds[1][1], "split"] = "val"
assert pool.groupby("group_id").split.nunique().max() == 1   # группа целиком в одном сплите

stats = pd.crosstab(pool.split, pool.label, margins=True, margins_name="all").loc[["train", "val", "test", "all"]]
stats.columns = ["label_0", "label_1", "total"]
stats["share_1"] = (stats.label_1 / stats.total).round(3)
stats

,label_0,label_1,total,share_1
split,,,,
train,2051,1327,3378,0.393
val,256,166,422,0.393
test,257,166,423,0.392
all,2564,1659,4223,0.393


## 8. Сохранение

In [14]:
cols = ["id", "source_id", "text", "label", "source", "domain", "rating",
        "label_source", "regex_label", "regex_terms", "group_id"]
for sp in ["train", "val", "test"]:
    pool[pool.split == sp][cols].to_csv(OUT / f"{sp}.csv", index=False)
stats.to_csv(OUT / "split_stats.csv")
print(list(OUT.iterdir()))

[PosixPath('data/val.csv'), PosixPath('data/train.csv'), PosixPath('data/split_stats.csv'), PosixPath('data/test.csv')]


## 9. Regex как baseline на test
v2 писался после просмотра размеченных данных (включая test), поэтому его оценка завышена; честный baseline — v1.

In [15]:
test = pool[pool.split == "test"]
y = test.label == 1
for v in [1, 2]:
    pred = test.text.map(lambda t: kz_regex.is_profane(t, v))
    tp, fp, fn = (pred & y).sum(), (pred & ~y).sum(), (~pred & y).sum()
    print(f"regex v{v}: P={tp/(tp+fp):.3f}  R={tp/(tp+fn):.3f}  F1={2*tp/(2*tp+fp+fn):.3f}")

regex v1: P=0.982  R=0.964  F1=0.973
regex v2: P=0.982  R=1.000  F1=0.991


## Дальше: модель
Любой словарь / токенизатор / TF-IDF обучать **только на train**:
```python
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))
X_train = vec.fit_transform(train.text)        # fit только на train
X_val, X_test = vec.transform(val.text), vec.transform(test.text)
```